# Halcyon - Aerospace Team Graz - 2023

Launched at EuRoC'23, allowing the team to become the Overall Champions.
Permission to use flight data given by Dorothea Krasser, 2024.

These results were extracted out of the flight card:

1. Team number: `1`
2. Launch date: `October 13th, 2023. around 14hrs local time`
3. Last simulated apogee before flight: `3163 m` (this value differs from the simulation shown below because of the updates to rocketpy software)
4. Official recorded apogee: `3450 m`

The relative error of altitude apogee is only `9.1%`


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime

import matplotlib.pyplot as plt
import numpy as np

from rocketpy import Environment, Flight, Function, Rocket
from rocketpy.motors import CylindricalTank, Fluid, HybridMotor
from rocketpy.motors.tank import MassFlowRateBasedTank
from rocketpy.tools import quaternions_to_spin
from rocketpy import Accelerometer, GnssReceiver, Gyroscope


plt.style.use("seaborn-v0_8-colorblind")

In [ ]:
tomorrow = datetime.date(2023, 10, 13) + datetime.timedelta(days=1)

Env = Environment(
    latitude=39.388692,
    longitude=-8.287814,
    elevation=130,
)

The team preferred to set a custom atmospheric model. However, for better resolution, this examples will only run the windy atmosphere that was registered in windy. 

In [ ]:
Env.set_date((tomorrow.year, tomorrow.month, tomorrow.day, 12))

Env.set_atmospheric_model(
    type="custom_atmosphere",
    pressure=None,
    temperature=300,
    wind_u=[(0, 8), (1000, 10)],
    wind_v=[(0, 0), (500, 0), (1600, 0)],
)

Env.max_expected_height = 5000

In [ ]:
Env.info()

## Environment

In [ ]:
env = Environment(
    gravity=9.80665,
    date=(2023, 10, 13, 14),
    latitude=39.388692,
    longitude=-8.287814,
    elevation=130,
    datum="WGS84",
    timezone="Portugal",
)

# env.set_atmospheric_model(type="Windy", file="ECMWF")
# env.max_expected_height = 4000
# env.info()

env.set_atmospheric_model(
    type="Reanalysis",
    file="../../data/weather/euroc_2023_all_windows.nc",
    dictionary="ECMWF",
)
env.max_expected_height = 4000
env.info()

## Motor

In [ ]:
oxidizer_liq = Fluid(name="N2O_l", density=960)
oxidizer_gas = Fluid(name="N2O_g", density=1.9277)

tank_shape = CylindricalTank(70 / 1000, 320 / 1000)

oxidizer_tank = MassFlowRateBasedTank(
    name="oxidizer_tank",
    geometry=tank_shape,
    flux_time=(5),
    initial_liquid_mass=4.3,
    initial_gas_mass=0,
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=4.2 / 5,
    gas_mass_flow_rate_in=0,
    gas_mass_flow_rate_out=0,
    liquid=oxidizer_liq,
    gas=oxidizer_gas,
)

hybrid_motor = HybridMotor(
    thrust_source="../../data/rockets/astg/engine_Halcyon_4thHotfire.eng",
    dry_mass=10670 / 1000,
    dry_inertia=(1.668, 1.668, 0.026),
    center_of_dry_mass_position=780 / 1000,
    burn_time=5,
    reshape_thrust_curve=False,
    grain_number=1,
    grain_separation=0,
    grain_outer_radius=43 / 1000,
    grain_initial_inner_radius=22.5 / 1000,
    grain_initial_height=310 / 1000,
    grain_density=920,
    nozzle_radius=0.0141,
    throat_radius=0.00677,
    interpolation_method="linear",
    grains_center_of_mass_position=385 / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

hybrid_motor.add_tank(tank=oxidizer_tank, position=934.75 / 1000)

In [ ]:
hybrid_motor.info()

## Rocket and Aerodynamic surfaces

In [ ]:
HALCYON = Rocket(
    radius=152.4 / 2000,
    mass=14613 / 1000,
    inertia=(24.56, 24.56, 70.074),
    center_of_mass_without_motor=2344 / 1000,
    power_off_drag="../../data/rockets/astg/DragCoeffOR_off.csv",
    power_on_drag="../../data/rockets/astg/DragCoeffOR_on.csv",
    coordinate_system_orientation="tail_to_nose",
)

HALCYON.set_rail_buttons(2.808, 1.549)

HALCYON.add_motor(hybrid_motor, position=20 / 1000)

In [ ]:
NoseCone = HALCYON.add_nose(length=0.46, kind="vonKarman", position=3556 / 1000)

FinSet = HALCYON.add_trapezoidal_fins(
    n=4,
    span=0.125,
    root_chord=0.247,
    tip_chord=0.045,
    position=0.263,
)

Tail = HALCYON.add_tail(
    top_radius=152.4 / 2000, bottom_radius=0.0496, length=0.254, position=0.254
)
HALCYON.info()

In [ ]:
# Remove parachute
# Main = HALCYON.add_parachute(
#     name="Main",
#     cd_s=9.621,
#     trigger="apogee",
#     sampling_rate=100,
#     lag=5,
# )
gyro_clean = Gyroscope(sampling_rate=100)
accelerometer_clean = Accelerometer(sampling_rate=100)
HALCYON.add_sensor(gyro_clean, position=1.5)
HALCYON.add_sensor(accelerometer_clean, position=1.5)
HALCYON.plots.draw()

In [ ]:
def tvc_controller_function(
    time, sampling_rate, state, state_history, observed_variables, tvc, sensors
):
    # state = [x, y, z, vx, vy, vz, e0, e1, e2, e3, wx, wy, wz]

    # print(time)

    tvc.gimbal_angle_x = 0
    tvc.gimbal_angle_y = 0
    # Return variables of interest to be saved in the observed_variables list
    return (
        time,
        tvc.gimbal_angle_x,
        tvc.gimbal_angle_y,
    )


tvc, tvc_controller = HALCYON.add_tvc(
    gimbal_range=10,
    sampling_rate=100,
    gimbal_rate_limit=100,
    controller_function=tvc_controller_function,
    return_controller=True,
)
tvc.info()
tvc_controller.info()

In [ ]:
def roll_controller_function(
    time, sampling_rate, state, state_history, observed_variables, roll_control, sensors
):
    # state = [x, y, z, vx, vy, vz, e0, e1, e2, e3, wx, wy, wz]

    # Separate sensor data by type
    gyro_data = None    # rad/s
    accel_data = None   # m/s^2
    
    for sensor in sensors:
        if isinstance(sensor, Gyroscope):
            sensor_time, gx, gy, gz = zip(*sensor.measured_data)
            gyro_data = {
                'time': np.array(sensor_time),
                'x': np.array(gx),
                'y': np.array(gy),
                'z': np.array(gz)
            }
        elif isinstance(sensor, Accelerometer):
            sensor_time, ax, ay, az = zip(*sensor.measured_data)
            accel_data = {
                'time': np.array(sensor_time),
                'x': np.array(ax),
                'y': np.array(ay),
                'z': np.array(az)
            }

    KP = 50
    KI = 1
    KD = 0.0

    # Roll rate target: 0.5 Hz sinusoidal, ±10 deg/s
    target = np.deg2rad(10 * np.sin(2 * np.pi * 0.5 * time)) # rad/s

    if gyro_data is None:
        roll_control.roll_torque = 0
        return (time, roll_control.roll_torque,)
    
    roll_rate_errors = target - gyro_data['z']
    roll_rate_error_integral = np.sum(roll_rate_errors) / sampling_rate
    roll_rate_error_derivative = (roll_rate_errors[-1] - roll_rate_errors[-2]) * sampling_rate if len(roll_rate_errors) > 1 else 0
    
    roll_control.roll_torque = KP * roll_rate_errors[-1] + KI * roll_rate_error_integral + KD * roll_rate_error_derivative
    
    # Return variables of interest to be saved in the observed_variables list
    return (
        time,
        roll_control.roll_torque,
    )


roll_control, roll_controller = HALCYON.add_roll_control(
    max_roll_torque=10,
    sampling_rate=100,
    torque_rate_limit=100,
    controller_function=roll_controller_function,
    return_controller=True,
)
roll_control.info()
roll_controller.info()

In [ ]:
def throttle_controller_function(
    time, sampling_rate, state, state_history, observed_variables, throttle_control, sensors
):
    # state = [x, y, z, vx, vy, vz, e0, e1, e2, e3, wx, wy, wz]

    # Throttle command: 0.5 Hz sinusoid, centered at 0.8, amplitude 0.2
    throttle_control.throttle = 0.8 + 0.2 * np.sin(2 * np.pi * 0.5 * time)

    # Return variables of interest to be saved in the observed_variables list
    return (
        time,
        throttle_control.throttle,
    )

In [ ]:
throttle_obj, throttle_controller = HALCYON.add_throttle_control(
    controller_function=throttle_controller_function,
    sampling_rate=100,
    throttle_range=(0.0, 1.0),
    throttle_rate_limit=100,
    initial_throttle=1.0,
    return_controller=True,
    clamp=True,
)
print("has add_throttle_control:", hasattr(HALCYON, "add_throttle_control"))
print("has throttle_control:", hasattr(HALCYON, "throttle_control"))
print("throttle_control:", getattr(HALCYON, "throttle_control", None))

## Flight Simulation Data

In [ ]:
test_flight = Flight(
    rocket=HALCYON,
    environment=env,
    inclination=85,
    heading=90,
    rail_length=12,
    time_overshoot=False,
    terminate_on_apogee=True,
    verbose=False,
)
# test_flight.plots.attitude_data()
# ===== 先看 throttle 有沒有真的變 =====
obs = np.array(throttle_controller.observed_variables)

t_cmd = obs[:, 0]
th_cmd = obs[:, 1]

plt.figure()
plt.plot(t_cmd, th_cmd, label="Throttle")
plt.xlabel("Time (s)")
plt.ylabel("Throttle")
plt.title("Throttle Command")
plt.legend()
plt.grid()
plt.show()

# ===== 再看推力有沒有跟著變 =====
plt.figure()
plt.plot(test_flight.net_thrust[:, 0], test_flight.net_thrust[:, 1], label="Effective Thrust")
plt.xlabel("Time (s)")
plt.ylabel("Thrust (N)")
plt.title("Effective Thrust")
plt.legend()
plt.grid()
plt.show()

time1, ax, ay, az = zip(*accelerometer_clean.measured_data)
time2, gx, gy, gz = zip(*gyro_clean.measured_data)

# plt.plot(time1, ax, label="Accelerometer X")
# plt.plot(time1, ay, label="Accelerometer Y")
# plt.plot(time1, az, label="Accelerometer Z")
# plt.xlabel("Time (s)")
# plt.ylabel("Acceleration (m/s^2)")
# plt.legend()
# plt.grid()
# plt.show()

plt.plot(time2, gx, label="Gyroscope X")
plt.plot(time2, gy, label="Gyroscope Y")
plt.plot(time2, gz, label="Gyroscope Z")
plt.xlabel("Time (s)")
plt.ylabel("Angular Velocity (rad/s)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
test_flight.prints.apogee_conditions()
test_flight.plots.trajectory_3d()
test_flight.plots.attitude_data()

test_flight.plots.linear_kinematics_data()
test_flight.plots.angular_kinematics_data()
